In [4]:

import json

import pandas as pd
import requests

In [1]:
API_URL = "http://localhost:8080/v1/chat/completions"


def extract_countries_llm(text: str) -> list:
    """
    Отправляет текст в LLM и возвращает список стран.
    """
    prompt = {
        "messages": [
            {
                "role": "system",
                "content": (
                    "Ты — NER-модель для извлечения стран."
                    "Если страна упомянута явно — запиши её. "
                    "Если страна не названа, но подразумевается по контексту "
                    "(например, «бригада Гивати» → Израиль, «сектор Газа» → Палестина), тоже добавь её. "
                    "Не включай в ответ города, фамилии, организации, альянсы, союзы, религии или географические регионы."
                    "Примеры того, что НЕ нужно возвращать: 'ООН', 'НАТО', 'ЕС', 'Путин', 'Москва', 'Евросоюз'."
                    "Возвращай только список стран мира, например ['Россия', 'США', 'Канада']."
                ),
            },
            {
                "role": "system",
                "content": "/no_think",
            },
            {
                "role": "user",
                "content": (
                    "Примеры:\n"
                    "Текст: Президент США посетил Канаду → [\"США\", \"Канада\"]\n"
                    "Текст: В Бейруте состоялся 14-й саммит Лиги арабских государств. → [\"Ливан\"]\n"
                    "Текст: В России состоялись выборы Президента Республики Ингушетия. → [\"Россия\"]"
                    f"Теперь обработай:\n{text}"
                ),
            },
        ],
        "max_tokens": 500,
        "temperature": 0.2,
        "top_p": 0.9
    }

    try:
        response = requests.post(API_URL, json=prompt, timeout=60)
        response.raise_for_status()
        data = response.json()

        # Извлекаем текст модели
        result = data["choices"][0]["message"]["content"]

        # Пытаемся преобразовать результат в список
        try:
            parsed = json.loads(result)
            if isinstance(parsed, list):
                return parsed
            else:
                # Если вернулась строка, например "США, Канада"
                return [c.strip() for c in parsed.strip("[]").replace('"', "").split(",")]
        except json.JSONDecodeError:
            # fallback: парсим вручную
            return [c.strip() for c in result.strip("[]").replace('"', "").split(",") if c.strip()]
    except Exception as e:
        print(f"⚠️ Ошибка при обработке текста: {e}")
        return []

In [7]:
example = "В аэропорту имени Чан Кайши при взлёте разбился Boeing 747 компании Singapore Airlines, погибли 83 из 179 человек на борту."
countries = extract_countries_llm(example)
print("🌍 Найденные страны:", countries)

🌍 Найденные страны: ['Сингапур']


In [8]:
df = pd.read_csv('../events/2_struct/2000-2025.csv')
df = df[:10]
texts = df["event"].tolist()
df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,Крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# === Путь для чекпоинта ===
SAVE_DIR = "../events/3_countries"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, "countries_progress.csv")

# === Загружаем прогресс, если он есть ===
if os.path.exists(SAVE_PATH):
    df = pd.read_csv(SAVE_PATH)
    print(f"🔄 Найден сохранённый прогресс: {SAVE_PATH}")
else:
    df = pd.read_csv("../events/2_struct/2000-2025.csv")
    if "countries" not in df.columns:
        df["countries"] = None
    print("🆕 Загружен новый датасет событий.")

# === Параметры ===
SAVE_EVERY = 10  # каждые 10 строк сохраняем чекпоинт

# === Индексы строк, где ещё нет результатов ===
pending = df["countries"].isna().to_numpy().nonzero()[0]

print(f"📊 Всего записей: {len(df)}, ещё не обработано: {len(pending)}")

# === Основной цикл обработки ===
for j, i in enumerate(tqdm(pending, total=len(pending), desc="Извлечение стран")):
    try:
        current_value = df.at[i, "countries"]
        if pd.notna(current_value):
            continue
        result = extract_countries_llm(df.at[i, "event"])
        df.at[i, "countries"] = result
    except Exception as e:
        df.at[i, "countries"] = f"ERROR: {e}"

    # === Сохраняем чекпоинт каждые N записей ===
    if (j + 1) % SAVE_EVERY == 0 or (j + 1) == len(pending):
        df.to_csv(SAVE_PATH, index=False, encoding="utf-8")
        print(f"💾 Промежуточное сохранение ({j + 1}/{len(pending)}): {SAVE_PATH}")

# === Финальное сохранение ===
df.to_csv(SAVE_PATH, index=False, encoding="utf-8")
print("🎉 Обработка завершена и сохранена:", SAVE_PATH)

🔄 Найден сохранённый прогресс: ../events/3_countries\countries_progress.csv
📊 Всего записей: 5644, ещё не обработано: 4849


Извлечение стран:   0%|          | 2/4849 [00:05<3:36:27,  2.68s/it]